# GridSense — LSTM Solar & Wind Forecaster (trained on real NASA POWER data)

Trains an LSTM (`LSTMForecaster`) that takes the **past 72 hours** of solar
irradiance, wind speed, and temperature and predicts the **next 48 hours** of
irradiance and wind speed.

## Why real data + why an LSTM

Earlier versions of this project's ML model were trained on a synthetic
weather generator — useful for proving the pipeline works end-to-end, but not
real evidence of forecasting skill. This notebook instead fetches **real,
multi-year hourly historical weather data from [NASA POWER](https://power.larc.nasa.gov/)**
— a free, no-API-key dataset covering any location on Earth back to 1981/1984
depending on parameter. Because the training sequences are now long and real
(years of actual hourly measurements, not one-shot snapshots), a **sequence
model (LSTM)** is the right tool — it can learn how irradiance and wind speed
actually evolve hour-to-hour at a given location, rather than just mapping a
single instant to an output.

## What "concrete proof" means here

The evaluation at the end of this notebook is a genuine **time-based
backtest**: the model is trained only on an earlier period and tested on a
**held-out later period it never saw during training** (not a random
shuffle-split, which would leak future information into training). The
reported MAE/RMSE are in real physical units (W/m² and km/h) on real,
unseen data for the chosen location. That is meaningfully different from
(and stronger than) anything in earlier versions of this project.

**What this still is NOT:** a forecast validated at *your* actual deployment
site with *your* actual sensors — it's validated against NASA's reanalysis
estimate for the coordinates you choose below. Reanalysis data is itself a
model output (blended satellite + ground-station + atmospheric modeling), not
a direct ground-truth measurement — a well-regarded one, but worth naming
plainly if a judge asks "concrete proof of what, exactly?"

## Architecture must match `backend/ml_model.py`

Same class name, same layer sizes/config, same `SEQ_LEN`/`HORIZON`/
normalization constants. That's what lets a `.pth` trained here load directly
into the backend with `load_state_dict` — no conversion step.

In [ ]:
import math
import time
from datetime import date

import numpy as np
import pandas as pd
import requests
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 1. Fetch real hourly weather data from NASA POWER

Free, no API key required. We request `ALLSKY_SFC_SW_DWN` (solar irradiance,
W/m²), `WS10M` (wind speed at 10m, m/s — converted to km/h to match the rest
of the project), and `T2M` (temperature, °C) for a chosen latitude/longitude.

NASA POWER's hourly endpoint limits how much date range a single request can
cover, so we fetch **one year at a time** and concatenate. Default location
below is Bhopal, India (matching the rest of the project) and a 5-year window
ending a couple of years back, chosen so the data is reliably already
published regardless of when you run this notebook. Change `LAT`/`LON` to
your actual site.

In [ ]:
LAT = 23.2599      # Bhopal, India — change to your site's coordinates
LON = 77.4126
START_YEAR = 2018
END_YEAR = 2022      # 5 years of hourly data — comfortably published by NASA POWER

NASA_POWER_URL = "https://power.larc.nasa.gov/api/temporal/hourly/point"
PARAMS = "ALLSKY_SFC_SW_DWN,WS10M,T2M"


def fetch_nasa_power_year(lat, lon, year, retries=3):
    start = f"{year}0101"
    end = f"{year}1231"
    params = {
        "parameters": PARAMS,
        "community": "RE",
        "longitude": lon,
        "latitude": lat,
        "start": start,
        "end": end,
        "format": "JSON",
    }
    for attempt in range(retries):
        resp = requests.get(NASA_POWER_URL, params=params, timeout=60)
        if resp.ok:
            return resp.json()["properties"]["parameter"]
        time.sleep(2 * (attempt + 1))
    resp.raise_for_status()


frames = []
for year in range(START_YEAR, END_YEAR + 1):
    print(f"Fetching {year}...")
    data = fetch_nasa_power_year(LAT, LON, year)
    df_year = pd.DataFrame(data)
    frames.append(df_year)

raw = pd.concat(frames)
raw.index = pd.to_datetime(raw.index, format="%Y%m%d%H")
raw = raw.sort_index()
raw.columns = ["irradiance", "wind_ms", "temperature_c"]

# NASA POWER uses -999 as a missing-value sentinel; interpolate small gaps
raw = raw.replace(-999.0, np.nan).interpolate(limit=6).dropna()
raw["wind_kmh"] = raw["wind_ms"] * 3.6

print(f"Fetched {len(raw)} hourly rows from {raw.index.min()} to {raw.index.max()}")
raw.head()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
sample = raw.iloc[:24*14]  # first 2 weeks
axes[0].plot(sample.index, sample["irradiance"], color='#F2A93B')
axes[0].set_ylabel("Irradiance (W/m²)")
axes[1].plot(sample.index, sample["wind_kmh"], color='#4FB6C7')
axes[1].set_ylabel("Wind speed (km/h)")
plt.suptitle("First two weeks of fetched data (sanity check)")
plt.tight_layout()
plt.show()

## 2. Build sliding-window sequences

Each training example:
- **Input**: the past `SEQ_LEN=72` hours of `(hour_sin, hour_cos, irradiance_norm, wind_norm, temp_norm)`
- **Target**: the next `HORIZON=48` hours of `(irradiance_norm, wind_norm)`

Normalization (must match `backend/ml_model.py` exactly):
- `irradiance_norm = irradiance / 1000`
- `wind_norm = wind_kmh / 100`
- `temp_norm = temperature_c / 40`
- `hour_sin`, `hour_cos` = sin/cos of hour-of-day, already in [-1, 1]

**Crucially, the train/test split is by TIME, not random** — training data is
everything before `SPLIT_DATE`, test data is everything after it. This is
what makes the later evaluation a genuine backtest instead of an optimistic,
leaky estimate.

In [ ]:
SEQ_LEN = 72   # hours of history — must match backend/ml_model.py
HORIZON = 48    # hours predicted forward — must match backend/ml_model.py

def normalize_frame(df):
    hours = df.index.hour.values
    hour_sin = np.sin(2 * np.pi * hours / 24)
    hour_cos = np.cos(2 * np.pi * hours / 24)
    irr_norm = np.clip(df["irradiance"].values / 1000.0, 0, 1)
    wind_norm = np.clip(df["wind_kmh"].values / 100.0, 0, 1)
    temp_norm = np.clip(df["temperature_c"].values / 40.0, 0, 1)
    features = np.stack([hour_sin, hour_cos, irr_norm, wind_norm, temp_norm], axis=1)
    targets = np.stack([irr_norm, wind_norm], axis=1)
    return features.astype(np.float32), targets.astype(np.float32)


def build_windows(features, targets, seq_len=SEQ_LEN, horizon=HORIZON):
    X, Y = [], []
    n = len(features)
    for t in range(seq_len, n - horizon):
        X.append(features[t - seq_len:t])
        Y.append(targets[t:t + horizon])
    return np.array(X, dtype=np.float32), np.array(Y, dtype=np.float32)


# Time-based split: last ~15% of the timeline is held out as a genuine backtest
SPLIT_DATE = raw.index[int(len(raw) * 0.85)]
print("Train/test split date:", SPLIT_DATE)

train_df = raw[raw.index < SPLIT_DATE]
test_df = raw[raw.index >= SPLIT_DATE]

train_feat, train_targ = normalize_frame(train_df)
test_feat, test_targ = normalize_frame(test_df)

X_train, Y_train = build_windows(train_feat, train_targ)
X_test, Y_test = build_windows(test_feat, test_targ)

print(f"Train windows: {X_train.shape}, Test windows: {X_test.shape}")

In [ ]:
class WindowDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.Y[i]


train_loader = DataLoader(WindowDataset(X_train, Y_train), batch_size=128, shuffle=True)
test_loader = DataLoader(WindowDataset(X_test, Y_test), batch_size=256, shuffle=False)

## 3. Model — must match `backend/ml_model.py` exactly

In [ ]:
INPUT_DIM = 5
HIDDEN_DIM = 64
NUM_LAYERS = 2
OUTPUT_DIM = 2

class LSTMForecaster(nn.Module):
    """Must be identical to the class of the same name in backend/ml_model.py."""
    def __init__(self, input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM,
                 num_layers=NUM_LAYERS, horizon=HORIZON, output_dim=OUTPUT_DIM):
        super().__init__()
        self.horizon = horizon
        self.output_dim = output_dim
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, horizon * output_dim),
            nn.Sigmoid(),
        )

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        last_hidden = h_n[-1]
        out = self.head(last_hidden)
        return out.view(-1, self.horizon, self.output_dim)


model = LSTMForecaster().to(device)
print(model)
print("Total parameters:", sum(p.numel() for p in model.parameters()))

## 4. Train

In [ ]:
EPOCHS = 25
LR = 1e-3

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.MSELoss()

train_losses, test_losses = [], []

for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        pred = model(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        optimizer.step()
        running += loss.item() * xb.size(0)
    train_loss = running / len(train_loader.dataset)

    model.eval()
    running = 0.0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            running += loss_fn(pred, yb).item() * xb.size(0)
    test_loss = running / len(test_loader.dataset)

    train_losses.append(train_loss)
    test_losses.append(test_loss)

    if epoch % 2 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{EPOCHS} — train MSE: {train_loss:.5f}  held-out MSE: {test_loss:.5f}")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label="train")
plt.plot(test_losses, label="held-out (later time period)")
plt.xlabel("Epoch"); plt.ylabel("MSE (normalized units)")
plt.title("Training curve — held-out set is a real future time period, never seen in training")
plt.legend()
plt.show()

## 5. Real backtest: MAE / RMSE in physical units

This is the number to actually cite as evidence of forecasting skill: error
computed on real NASA POWER data from a time period **after** the training
cutoff, denormalized back to W/m² and km/h.

For context, we also compute the error of a **naive persistence baseline**
(predict "next 48h = same as the last known hour, following the clear-sky
diurnal shape") on the same held-out data — if the LSTM doesn't beat this,
it isn't earning its complexity.

In [ ]:
def denormalize(arr_irr_norm, arr_wind_norm):
    return arr_irr_norm * 1000.0, arr_wind_norm * 100.0


model.eval()
all_pred, all_true = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        pred = model(xb.to(device)).cpu().numpy()
        all_pred.append(pred)
        all_true.append(yb.numpy())

all_pred = np.concatenate(all_pred)   # (N, horizon, 2)
all_true = np.concatenate(all_true)

pred_irr, pred_wind = denormalize(all_pred[..., 0], all_pred[..., 1])
true_irr, true_wind = denormalize(all_true[..., 0], all_true[..., 1])

mae_irr = np.mean(np.abs(pred_irr - true_irr))
rmse_irr = np.sqrt(np.mean((pred_irr - true_irr) ** 2))
mae_wind = np.mean(np.abs(pred_wind - true_wind))
rmse_wind = np.sqrt(np.mean((pred_wind - true_wind) ** 2))

print("== LSTM — held-out real backtest ==")
print(f"Irradiance  MAE: {mae_irr:.1f} W/m²   RMSE: {rmse_irr:.1f} W/m²")
print(f"Wind speed  MAE: {mae_wind:.2f} km/h   RMSE: {rmse_wind:.2f} km/h")

In [ ]:
# Naive persistence baseline on the same held-out windows, for honest comparison
def clear_sky_irradiance(hour_of_day):
    x = np.maximum(0.0, np.sin(((hour_of_day - 6.0) / 12.0) * np.pi))
    return 1000.0 * (x ** 1.1)


baseline_pred_irr, baseline_pred_wind, baseline_true_irr, baseline_true_wind = [], [], [], []

test_hours = test_df.index.hour.values
test_irr = test_df["irradiance"].values
test_wind = test_df["wind_kmh"].values

for t in range(SEQ_LEN, len(test_df) - HORIZON):
    last_hour = test_hours[t - 1]
    last_irr = test_irr[t - 1]
    last_wind = test_wind[t - 1]
    clear_now = clear_sky_irradiance(last_hour)
    attenuation = np.clip(last_irr / clear_now, 0, 1) if clear_now > 1 else 1.0

    future_hours = (last_hour + np.arange(1, HORIZON + 1)) % 24
    baseline_pred_irr.append(clear_sky_irradiance(future_hours) * attenuation)
    baseline_pred_wind.append(np.full(HORIZON, last_wind))
    baseline_true_irr.append(test_irr[t:t + HORIZON])
    baseline_true_wind.append(test_wind[t:t + HORIZON])

baseline_pred_irr = np.array(baseline_pred_irr)
baseline_pred_wind = np.array(baseline_pred_wind)
baseline_true_irr = np.array(baseline_true_irr)
baseline_true_wind = np.array(baseline_true_wind)

base_mae_irr = np.mean(np.abs(baseline_pred_irr - baseline_true_irr))
base_rmse_irr = np.sqrt(np.mean((baseline_pred_irr - baseline_true_irr) ** 2))
base_mae_wind = np.mean(np.abs(baseline_pred_wind - baseline_true_wind))
base_rmse_wind = np.sqrt(np.mean((baseline_pred_wind - baseline_true_wind) ** 2))

print("== Persistence baseline — same held-out period ==")
print(f"Irradiance  MAE: {base_mae_irr:.1f} W/m²   RMSE: {base_rmse_irr:.1f} W/m²")
print(f"Wind speed  MAE: {base_mae_wind:.2f} km/h   RMSE: {base_rmse_wind:.2f} km/h")

print("\n== Comparison ==")
print(f"Irradiance MAE improvement: {(1 - mae_irr/base_mae_irr)*100:+.1f}%")
print(f"Wind speed MAE improvement: {(1 - mae_wind/base_mae_wind)*100:+.1f}%")

**Read this comparison honestly.** If the LSTM's improvement over persistence
is small or negative for your chosen location, that's a real and useful
finding — it means the extra model complexity isn't paying for itself there,
and persistence (which is far simpler and cheaper to run) may be the better
choice. Report whichever number you actually get, not the one you hoped for.

## 6. Plot one example forecast vs. reality

In [ ]:
idx = len(X_test) // 2  # a window from the middle of the held-out period
sample_x = torch.tensor(X_test[idx:idx+1], dtype=torch.float32).to(device)
with torch.no_grad():
    sample_pred = model(sample_x)[0].cpu().numpy()
sample_true = Y_test[idx]

hours_ahead = np.arange(1, HORIZON + 1)
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

axes[0].plot(hours_ahead, sample_true[:, 0] * 1000, label="true (NASA POWER)", color='#8FA3B5')
axes[0].plot(hours_ahead, sample_pred[:, 0] * 1000, label="LSTM prediction", color='#F2A93B', linestyle='--')
axes[0].set_ylabel("Irradiance (W/m²)"); axes[0].legend()

axes[1].plot(hours_ahead, sample_true[:, 1] * 100, label="true (NASA POWER)", color='#8FA3B5')
axes[1].plot(hours_ahead, sample_pred[:, 1] * 100, label="LSTM prediction", color='#4FB6C7', linestyle='--')
axes[1].set_ylabel("Wind speed (km/h)"); axes[1].set_xlabel("Hours ahead"); axes[1].legend()

plt.suptitle("Example 48h forecast vs. real held-out NASA POWER data")
plt.tight_layout()
plt.show()

## 7. Save weights

Saves just the `state_dict` — what `backend/ml_model.py`'s `load_model()`
expects.

In [ ]:
import os

WEIGHTS_FILENAME = "solar_wind_forecaster.pth"
torch.save(model.state_dict(), WEIGHTS_FILENAME)
print(f"Saved {WEIGHTS_FILENAME} ({os.path.getsize(WEIGHTS_FILENAME) / 1024:.1f} KB)")

## 8. Download the weights

- **Drop it in the project:** place it at `backend/weights/solar_wind_forecaster.pth`
  and (re)start the backend.
- **Upload through the running app:** in the frontend's "3D sensor simulation"
  panel, set "Forecast method" to "Trained ML model" and use the upload
  control — it POSTs to `/api/model/upload-weights` and loads it immediately.

In [ ]:
try:
    from google.colab import files
    files.download(WEIGHTS_FILENAME)
except ImportError:
    print("Not running in Colab — find the file at:", os.path.abspath(WEIGHTS_FILENAME))

## Notes on generalizing / improving this further

- **This model is location-specific.** It was trained on weather patterns for
  the one `(LAT, LON)` set above. Using it for a very different climate will
  likely perform worse than the backtest numbers here suggest — retrain per
  site, or add latitude/longitude as model inputs and train on many sites at
  once if you need one model to generalize geographically.
- **The backend's live history buffer starts empty.** Until real sensor
  readings (or repeated 3D-twin interaction) have populated at least
  `SEQ_LEN=72` hours of history, `source=ml` forecasts in the running app are
  partly built on padded/extrapolated history, not the real sequences this
  notebook trains and evaluates on — the API response's `sensor_meta` field
  reports exactly how many of the input hours were real vs. padded.
- **Reanalysis vs. ground truth:** NASA POWER is itself a modeled estimate
  (satellite + station + atmospheric-model blend), not direct ground
  measurement. For a real deployment, validate the final model against your
  own site's actual sensor logs once you have enough of them, and consider
  retraining on that real local data entirely.